In [0]:
%pip install pymysql

In [0]:
from pyspark.sql.functions import *
from framework.reader import *
from framework.writer import *
from framework.audit import *
from framework.validator import *
from framework.metrics import *
from framework.mysql_connector import *
from framework.transformations.erp.erp_transform import *
from framework.transformations.crm.crm_transform import *
from framework.transformations.inventory.inventory_transform import *
from framework.transformations.website.website_transform import *
from framework.transformations.payments.payments_transform import *

In [0]:
TRANSFORMATIONS={
    "build_product_dimension": build_product_dimension,
    "build_category_dimension": build_category_dimension,
    "build_supplier_dimension": build_supplier_dimension,
    "build_customer_dimension": build_customer_dimension,
    "build_warehouse_dimension": build_warehouse_dimension,
    "build_inventory_fact": build_inventory_fact,
    "build_orders_fact": build_orders_fact,
    "build_order_items_fact": build_order_items_fact,
    "build_payment_fact": build_payment_fact
}

In [0]:
silver_pipelines=fetch_eligible_silver_pipelines()
if not silver_pipelines:
  print("No silver pipelines found")
else:
  for pipeline in silver_pipelines:
    try:
       dedup_used=pipeline["dedup_used"]
       explode_used=pipeline["explode_used"]
       silver_batch_id=None
       input_record=0
       output_record=0
       duplicate_removed_record=0
       print(f"Executing silver pipeline:{pipeline["target_table"]}")
       silver_batch_id=insert_silver_execution(pipeline["target_table"])
       bronze_df=read_delta(spark, pipeline['source_path'])
       transform_fun=TRANSFORMATIONS[pipeline['transformation_fun']]
       silver_df,input_record,output_record,duplicate_removed_record=transform_fun(bronze_df)
       write_delta(silver_df,pipeline["target_path"],pipeline["write_mode"])
       silver_update_completed(silver_batch_id,input_record,output_record,duplicate_removed_record,dedup_used,explode_used)
       print(f"Completed silver pipeline:{pipeline["target_table"]}")
    except Exception as e:
       if silver_batch_id is not None:
          silver_update_failed(silver_batch_id,input_record,output_record,duplicate_removed_record,str(e)[:1000],dedup_used,explode_used)
       else:
          print(f"Ingestion of pipeline:{pipeline['target_table']} failed")